# 06 — Regression-Based Forecasting

**Gold Nexus Alpha — professor-style forecasting pipeline**

Purpose of this notebook:

1. Load the core multivariate dataset created by Notebook 03.
2. Use the locked professor-safe Dataset B window.
3. Convert `date` with `pd.to_datetime(...)` and use date-indexed time-series data.
4. Plot the gold time series before modeling.
5. Create a professor-style regression trend benchmark.
6. Build a selected-factor OLS regression model for gold forecasting.
7. Compare validation and test accuracy using proper time-based splits.
8. Export regression metrics, diagnostics, coefficient tables, and page JSON artifacts.
9. Push outputs to GitHub.

Professor reference mirrored from the uploaded regression file:

- `Chapter 17 - Regression-based forecasting (2).ipynb`

Professor-style elements mirrored:

- `pd.to_datetime(...)`
- `pd.Series(values, index=date, name=...)`
- `statsmodels.formula.api` OLS formula workflow
- time-series plot before modeling
- time-based train / validation / test partitioning
- trend benchmark regression
- actual vs fitted / forecast chart
- residual / forecast error chart
- regression summary table
- coefficient, t-value, p-value, R², adjusted R², F-test reporting

Project-safe regression rule:

- Main regression uses the locked core multivariate dataset.
- `high_yield` is excluded because it starts too late.
- The notebook uses selected predictors only, not every factor blindly.
- Validation forecasts are trained only on the training period.
- Test forecasts are trained on training + validation only.
- The test period is never used to fit coefficients.


In [ ]:
# ======================================================================================
# CELL 1 — REPO SYNC AND CLEAN RESET
# ======================================================================================
# Purpose:
# - Clone the GitHub repository into Colab.
# - Load GITHUB_TOKEN from Colab Secrets.
# - Keep the same clean Colab → GitHub artifact workflow as Notebooks 01–05.
# ======================================================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
except Exception:
    userdata = None

REPO_OWNER = "rathee000001"
REPO_NAME  = "nyit-gold-intelligence-2026"
BRANCH     = "main"

BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / REPO_NAME

GITHUB_TOKEN = None
if userdata is not None:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

def run_cmd(cmd, cwd=None, allow_fail=False, display_cmd=None):
    """Run a shell command without printing secrets."""
    shown = display_cmd if display_cmd is not None else cmd
    if isinstance(shown, (list, tuple)):
        shown = " ".join(str(x) for x in shown)
    print(f">> {shown}")
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {shown}")
    return p

# Clean reset: remove old clone if present.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
    print(f"🧹 Removed existing project folder: {PROJECT_DIR}")

# Clone URL. Use token for private/push access, but never print the token.
public_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git" if GITHUB_TOKEN else public_url

run_cmd(
    ["git", "clone", "--branch", BRANCH, auth_url, str(PROJECT_DIR)],
    display_cmd=["git", "clone", "--branch", BRANCH, f"https://***@github.com/{REPO_OWNER}/{REPO_NAME}.git", str(PROJECT_DIR)]
)

# Git identity for Colab commits.
run_cmd(["git", "config", "user.email", "colab-artifact-bot@gold-nexus-alpha.local"], cwd=str(PROJECT_DIR))
run_cmd(["git", "config", "user.name", "Gold Nexus Alpha Colab"], cwd=str(PROJECT_DIR))

# Ensure we are clean and up to date.
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

print("✅ Repo ready:", PROJECT_DIR)
print("✅ Branch:", BRANCH)
print("✅ UTC time:", datetime.now(timezone.utc).isoformat())


In [ ]:
# ======================================================================================
# CELL 2 — DEPENDENCIES
# ======================================================================================
# Purpose:
# - Load the same regression/time-series libraries used in the professor's Chapter 17 notebook.
# - Install statsmodels only if the Colab runtime does not already have it.
# ======================================================================================

import sys
import json
import math
import glob
import warnings
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import statsmodels
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa import tsatools, stattools
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_ljungbox

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

%matplotlib inline

print("✅ Dependencies loaded")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("statsmodels:", statsmodels.__version__)


In [ ]:
# ======================================================================================
# CELL 3 — PATH SETUP, INPUT DETECTION, AND LOCKED TIME WINDOWS
# ======================================================================================
# Purpose:
# - Locate Notebook 03 output: data/aligned/model_ready_multivariate.csv
# - Fall back to weekday_clean_matrix.csv or an uploaded Gold_Matrix CSV only if needed.
# - Lock the professor-safe core multivariate dataset and time splits.
# ======================================================================================

PROJECT_DIR = Path(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
ALIGNED_DIR = DATA_DIR / "aligned"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
MODELS_ARTIFACTS_DIR = ARTIFACTS_DIR / "models"
INTERPRETABILITY_ARTIFACTS_DIR = ARTIFACTS_DIR / "interpretability"
PAGES_ARTIFACTS_DIR = ARTIFACTS_DIR / "pages"

for folder in [ALIGNED_DIR, MODELS_ARTIFACTS_DIR, INTERPRETABILITY_ARTIFACTS_DIR, PAGES_ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Locked project rules.
OFFICIAL_FORECAST_CUTOFF_DATE = "2026-03-31"

CORE_MULTIVARIATE_START = "2006-01-02"
CORE_MULTIVARIATE_END   = OFFICIAL_FORECAST_CUTOFF_DATE

TRAIN_START = "2006-01-02"
TRAIN_END   = "2018-12-31"

VALIDATION_START = "2019-01-01"
VALIDATION_END   = "2022-12-30"

TEST_START = "2023-01-02"
TEST_END   = "2026-03-31"

TARGET_COL = "gold_price"

# Professor-readable selected predictors from the locked project plan.
REQUESTED_REGRESSION_PREDICTORS = [
    "gold_lag_1",
    "gold_ma_20",
    "real_yield",
    "nominal_yield",
    "usd_index",
    "vix_index",
    "fin_stress",
    "gpr_index",
    "policy_unc",
    "oil_wti",
    "gld_tonnes",
    "unrate",
]

EXCLUDED_FROM_MAIN_REGRESSION = ["high_yield"]

candidate_inputs = [
    ALIGNED_DIR / "model_ready_multivariate.csv",
    ALIGNED_DIR / "weekday_clean_matrix.csv",
    PROJECT_DIR / "Gold_Matrix_M3_Daily_2026-04-30.csv",
]

# Colab upload fallback, useful if the repo does not yet contain the CSV.
candidate_inputs += sorted(Path("/content").glob("*Gold*Matrix*.csv"))
candidate_inputs += sorted(Path("/content").glob("*gold*matrix*.csv"))

INPUT_PATH = None
for path in candidate_inputs:
    if path.exists():
        INPUT_PATH = path
        break

if INPUT_PATH is None:
    raise FileNotFoundError(
        "Could not find model_ready_multivariate.csv, weekday_clean_matrix.csv, or an uploaded Gold_Matrix CSV. "
        "Run Notebook 03 first or upload the current matrix CSV."
    )

print("✅ Input detected:", INPUT_PATH)

raw_df = pd.read_csv(INPUT_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

if "date" not in raw_df.columns:
    raise ValueError("Input file must contain a 'date' column.")

if TARGET_COL not in raw_df.columns:
    raise ValueError(f"Input file must contain target column '{TARGET_COL}'.")

raw_df["date"] = pd.to_datetime(raw_df["date"], errors="coerce")
raw_df = raw_df.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

# If the fallback source is the clean matrix instead of Notebook 03 output, create only the engineered
# gold features needed for this regression notebook. Shifted features avoid using the current target value.
if "gold_lag_1" not in raw_df.columns:
    raw_df["gold_lag_1"] = raw_df[TARGET_COL].shift(1)

if "gold_ma_20" not in raw_df.columns:
    raw_df["gold_ma_20"] = raw_df[TARGET_COL].rolling(window=20, min_periods=20).mean().shift(1)

# Enforce the locked core multivariate date window.
core_df = raw_df[
    (raw_df["date"] >= pd.Timestamp(CORE_MULTIVARIATE_START)) &
    (raw_df["date"] <= pd.Timestamp(CORE_MULTIVARIATE_END))
].copy()

if "high_yield" in core_df.columns:
    print("✅ high_yield detected but excluded from main regression by project rule.")

missing_predictors = [c for c in REQUESTED_REGRESSION_PREDICTORS if c not in core_df.columns]
if missing_predictors:
    raise ValueError(
        "The following requested regression predictors are missing. "
        "Run Notebook 03 again or check the source matrix: "
        + ", ".join(missing_predictors)
    )

REGRESSION_PREDICTORS = [c for c in REQUESTED_REGRESSION_PREDICTORS if c in core_df.columns and c not in EXCLUDED_FROM_MAIN_REGRESSION]

# Keep only required model columns.
model_cols = ["date", TARGET_COL] + REGRESSION_PREDICTORS
model_df_before_dropna = core_df[model_cols].copy()

rows_before_dropna = len(model_df_before_dropna)
model_df = model_df_before_dropna.dropna(subset=[TARGET_COL] + REGRESSION_PREDICTORS).copy()
rows_after_dropna = len(model_df)
rows_dropped_missing = rows_before_dropna - rows_after_dropna

# Date-indexed time-series frame, matching professor style.
model_df = model_df.sort_values("date").set_index("date")
model_df.index = pd.DatetimeIndex(model_df.index)

# Add professor-style trend variable like Chapter 17.
model_df["trend"] = np.arange(1, len(model_df) + 1)
model_df["trend_sq"] = np.square(model_df["trend"])
model_df["month"] = model_df.index.month

gold_price_ts = pd.Series(model_df[TARGET_COL].values, index=model_df.index, name=TARGET_COL)

train_df = model_df.loc[TRAIN_START:TRAIN_END].copy()
valid_df = model_df.loc[VALIDATION_START:VALIDATION_END].copy()
test_df  = model_df.loc[TEST_START:TEST_END].copy()
train_valid_df = model_df.loc[TRAIN_START:VALIDATION_END].copy()

if train_df.empty or valid_df.empty or test_df.empty:
    raise ValueError("One or more locked time splits are empty. Check date parsing and cutoff rules.")

print("✅ Core multivariate regression dataset ready")
print("Input rows before core window:", len(raw_df))
print("Core window rows before missing drop:", rows_before_dropna)
print("Core window rows after missing drop:", rows_after_dropna)
print("Rows dropped due to missing selected model values:", rows_dropped_missing)
print("Selected predictors:", REGRESSION_PREDICTORS)
print("Train:", train_df.index.min().date(), "to", train_df.index.max().date(), "| rows:", len(train_df))
print("Validation:", valid_df.index.min().date(), "to", valid_df.index.max().date(), "| rows:", len(valid_df))
print("Test:", test_df.index.min().date(), "to", test_df.index.max().date(), "| rows:", len(test_df))

display(model_df.head())

# Professor-style plot before modeling.
ax = gold_price_ts.plot(figsize=(12, 4), linewidth=0.8, title="Gold Price Time Series — Core Regression Window")
ax.set_xlabel("Date")
ax.set_ylabel("Gold Price")
plt.show()


In [ ]:
# ======================================================================================
# CELL 4 — MAIN REGRESSION MODELING LOGIC
# ======================================================================================
# Purpose:
# - Mirror professor Chapter 17 flow: series → trend variables → partition → OLS → forecast → residuals → accuracy.
# - Use selected predictors only for the main regression model.
# - Keep validation and test fitting leakage-safe.
# ======================================================================================

# --------------------------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------------------------

def to_float_or_none(x):
    if x is None:
        return None
    try:
        if pd.isna(x) or np.isinf(x):
            return None
        return float(x)
    except Exception:
        return None

def safe_mape(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = actual != 0
    if mask.sum() == 0:
        return None
    return float(np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100)

def directional_accuracy(actual, pred, prior_actual):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    prior_actual = np.asarray(prior_actual, dtype=float)
    mask = np.isfinite(actual) & np.isfinite(pred) & np.isfinite(prior_actual)
    if mask.sum() == 0:
        return None
    actual_direction = np.sign(actual[mask] - prior_actual[mask])
    pred_direction = np.sign(pred[mask] - prior_actual[mask])
    return float(np.mean(actual_direction == pred_direction) * 100)

def metric_dict(actual, pred, prior_actual=None):
    actual = pd.Series(actual).astype(float)
    pred = pd.Series(pred).astype(float)
    residual = actual - pred

    metrics = {
        "n": int(len(actual)),
        "mae": to_float_or_none(np.mean(np.abs(residual))),
        "mse": to_float_or_none(np.mean(np.square(residual))),
        "rmse": to_float_or_none(np.sqrt(np.mean(np.square(residual)))),
        "mape": to_float_or_none(safe_mape(actual, pred)),
        "bias_mean_error": to_float_or_none(np.mean(residual)),
        "mean_actual": to_float_or_none(np.mean(actual)),
        "mean_prediction": to_float_or_none(np.mean(pred)),
    }

    if prior_actual is not None:
        metrics["directional_accuracy_pct"] = to_float_or_none(directional_accuracy(actual, pred, prior_actual))

    return metrics

def coefficient_table(fit):
    rows = []
    conf_int = fit.conf_int()
    for term in fit.params.index:
        p_value = fit.pvalues.get(term, np.nan)
        coef = fit.params.get(term, np.nan)
        rows.append({
            "term": str(term),
            "coefficient": to_float_or_none(coef),
            "std_error": to_float_or_none(fit.bse.get(term, np.nan)),
            "t_value": to_float_or_none(fit.tvalues.get(term, np.nan)),
            "p_value": to_float_or_none(p_value),
            "conf_low_95": to_float_or_none(conf_int.loc[term, 0]) if term in conf_int.index else None,
            "conf_high_95": to_float_or_none(conf_int.loc[term, 1]) if term in conf_int.index else None,
            "significant_at_0_05": bool(pd.notna(p_value) and p_value < 0.05),
            "direction": "positive" if pd.notna(coef) and coef > 0 else ("negative" if pd.notna(coef) and coef < 0 else "zero_or_unknown"),
        })
    return rows

def fit_stats(fit):
    return {
        "r_squared": to_float_or_none(fit.rsquared),
        "adjusted_r_squared": to_float_or_none(fit.rsquared_adj),
        "f_statistic": to_float_or_none(fit.fvalue),
        "f_p_value": to_float_or_none(fit.f_pvalue),
        "aic": to_float_or_none(fit.aic),
        "bic": to_float_or_none(fit.bic),
        "df_model": to_float_or_none(fit.df_model),
        "df_resid": to_float_or_none(fit.df_resid),
        "condition_number": to_float_or_none(fit.condition_number),
        "nobs": int(fit.nobs),
    }

def split_prior_actual(split_df):
    # Previous observed value available before each row. This is used only for direction accuracy.
    prior = model_df[TARGET_COL].shift(1).reindex(split_df.index)
    return prior

def build_path_records(split_name, df, pred):
    out = pd.DataFrame({
        "date": df.index.astype(str),
        "split": split_name,
        "actual": df[TARGET_COL].values,
        "predicted": np.asarray(pred, dtype=float),
    })
    out["residual"] = out["actual"] - out["predicted"]
    out["absolute_error"] = np.abs(out["residual"])
    out["absolute_percentage_error"] = np.where(
        out["actual"] != 0,
        np.abs(out["residual"] / out["actual"]) * 100,
        np.nan
    )
    return out

def residual_summary(residuals):
    residuals = pd.Series(residuals).dropna().astype(float)
    return {
        "n": int(len(residuals)),
        "mean": to_float_or_none(residuals.mean()),
        "std": to_float_or_none(residuals.std()),
        "min": to_float_or_none(residuals.min()),
        "p25": to_float_or_none(residuals.quantile(0.25)),
        "median": to_float_or_none(residuals.median()),
        "p75": to_float_or_none(residuals.quantile(0.75)),
        "max": to_float_or_none(residuals.max()),
    }

def compute_vif(df, predictors):
    vif_rows = []
    clean_x = df[predictors].replace([np.inf, -np.inf], np.nan).dropna()
    if clean_x.empty or len(predictors) < 2:
        return vif_rows

    x = sm.add_constant(clean_x)
    for i, column in enumerate(x.columns):
        if column == "const":
            continue
        try:
            vif_value = variance_inflation_factor(x.values, i)
        except Exception:
            vif_value = np.nan
        vif_rows.append({
            "feature": str(column),
            "vif": to_float_or_none(vif_value),
            "flag": "high_multicollinearity_watch" if pd.notna(vif_value) and vif_value >= 10 else "ok_or_moderate",
        })
    return vif_rows

def ljung_box_records(residuals, lags=(10, 20)):
    residuals = pd.Series(residuals).dropna().astype(float)
    safe_lags = [lag for lag in lags if lag < len(residuals) - 1]
    if not safe_lags:
        return []
    try:
        lb = acorr_ljungbox(residuals, lags=safe_lags, return_df=True)
        return [
            {
                "lag": int(idx),
                "lb_stat": to_float_or_none(row["lb_stat"]),
                "p_value": to_float_or_none(row["lb_pvalue"]),
            }
            for idx, row in lb.iterrows()
        ]
    except Exception as exc:
        return [{"error": str(exc)}]

def fit_regression_candidate(model_id, model_name, formula, selected_predictors):
    # Validation fit: train only.
    validation_fit = smf.ols(formula=formula, data=train_df).fit()
    train_pred_from_validation_fit = validation_fit.predict(train_df)
    valid_pred = validation_fit.predict(valid_df)

    # Test fit: train + validation only.
    test_fit = smf.ols(formula=formula, data=train_valid_df).fit()
    train_valid_pred_from_test_fit = test_fit.predict(train_valid_df)
    test_pred = test_fit.predict(test_df)

    train_metrics = metric_dict(
        train_df[TARGET_COL],
        train_pred_from_validation_fit,
        prior_actual=split_prior_actual(train_df)
    )
    valid_metrics = metric_dict(
        valid_df[TARGET_COL],
        valid_pred,
        prior_actual=split_prior_actual(valid_df)
    )
    test_metrics = metric_dict(
        test_df[TARGET_COL],
        test_pred,
        prior_actual=split_prior_actual(test_df)
    )

    return {
        "model_id": model_id,
        "model_name": model_name,
        "formula": formula,
        "selected_predictors": selected_predictors,
        "validation_fit": validation_fit,
        "test_fit": test_fit,
        "train_prediction": train_pred_from_validation_fit,
        "validation_prediction": valid_pred,
        "train_valid_prediction": train_valid_pred_from_test_fit,
        "test_prediction": test_pred,
        "metrics": {
            "train_from_train_fit": train_metrics,
            "validation_from_train_fit": valid_metrics,
            "test_from_train_validation_fit": test_metrics,
        },
        "fit_statistics_validation_fit": fit_stats(validation_fit),
        "fit_statistics_test_fit": fit_stats(test_fit),
    }

# --------------------------------------------------------------------------------------
# Professor-style model formulas
# --------------------------------------------------------------------------------------

selected_formula_rhs = " + ".join(REGRESSION_PREDICTORS)

regression_candidates = []

# Chapter 17 style benchmark: trend-only regression.
regression_candidates.append(
    fit_regression_candidate(
        model_id="trend_only_ols",
        model_name="Trend-Only OLS Benchmark",
        formula=f"{TARGET_COL} ~ trend",
        selected_predictors=["trend"],
    )
)

# Main project regression: selected predictors only.
regression_candidates.append(
    fit_regression_candidate(
        model_id="selected_factor_ols",
        model_name="Selected-Factor OLS Regression",
        formula=f"{TARGET_COL} ~ {selected_formula_rhs}",
        selected_predictors=REGRESSION_PREDICTORS,
    )
)

# Professor-safe comparison candidate: selected predictors plus a simple trend term.
regression_candidates.append(
    fit_regression_candidate(
        model_id="selected_factor_plus_trend_ols",
        model_name="Selected-Factor OLS Regression + Trend",
        formula=f"{TARGET_COL} ~ trend + {selected_formula_rhs}",
        selected_predictors=["trend"] + REGRESSION_PREDICTORS,
    )
)

candidate_summary_rows = []
for candidate in regression_candidates:
    row = {
        "model_id": candidate["model_id"],
        "model_name": candidate["model_name"],
        "formula": candidate["formula"],
        "validation_rmse": candidate["metrics"]["validation_from_train_fit"]["rmse"],
        "validation_mae": candidate["metrics"]["validation_from_train_fit"]["mae"],
        "validation_mape": candidate["metrics"]["validation_from_train_fit"]["mape"],
        "test_rmse": candidate["metrics"]["test_from_train_validation_fit"]["rmse"],
        "test_mae": candidate["metrics"]["test_from_train_validation_fit"]["mae"],
        "test_mape": candidate["metrics"]["test_from_train_validation_fit"]["mape"],
        "adj_r_squared_test_fit": candidate["fit_statistics_test_fit"]["adjusted_r_squared"],
    }
    candidate_summary_rows.append(row)

candidate_summary_df = pd.DataFrame(candidate_summary_rows).sort_values("validation_rmse").reset_index(drop=True)

# Pick the best validation candidate within the selected-factor regression family.
# The trend-only model is retained as a professor-style benchmark, not as the main project regression.
eligible_main_candidates = candidate_summary_df[candidate_summary_df["model_id"] != "trend_only_ols"].copy()
MAIN_REGRESSION_MODEL_ID = eligible_main_candidates.sort_values("validation_rmse").iloc[0]["model_id"]
main_candidate = next(c for c in regression_candidates if c["model_id"] == MAIN_REGRESSION_MODEL_ID)

print("✅ Regression candidates completed")
display(candidate_summary_df)

print("✅ Main regression candidate for this notebook:", MAIN_REGRESSION_MODEL_ID)
print("Formula:", main_candidate["formula"])

# Professor-style regression summary table.
print("\n==================== REGRESSION SUMMARY: MAIN CANDIDATE, TRAIN + VALIDATION FIT ====================")
print(main_candidate["test_fit"].summary())

# --------------------------------------------------------------------------------------
# Forecast path and residual records for the main candidate
# --------------------------------------------------------------------------------------

path_train = build_path_records("train", train_df, main_candidate["train_prediction"])
path_valid = build_path_records("validation", valid_df, main_candidate["validation_prediction"])
path_test  = build_path_records("test", test_df, main_candidate["test_prediction"])
regression_forecast_path_df = pd.concat([path_train, path_valid, path_test], ignore_index=True)

# Residual diagnostics use the train + validation fit because that is the model used to forecast test.
train_valid_residuals = train_valid_df[TARGET_COL] - main_candidate["train_valid_prediction"]
test_residuals = test_df[TARGET_COL] - main_candidate["test_prediction"]

residual_acf_values = stattools.acf(pd.Series(train_valid_residuals).dropna(), nlags=20, fft=True)
residual_acf_records = [
    {"lag": int(i), "acf": to_float_or_none(v)}
    for i, v in enumerate(residual_acf_values)
]

vif_table = compute_vif(train_valid_df, main_candidate["selected_predictors"])

# --------------------------------------------------------------------------------------
# Professor-style charts
# --------------------------------------------------------------------------------------

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 8), sharex=False)

# Actual vs fitted / forecast.
train_df[TARGET_COL].plot(ax=axes[0], linewidth=0.7, label="Train Actual")
valid_df[TARGET_COL].plot(ax=axes[0], linewidth=0.7, linestyle="dashed", label="Validation Actual")
test_df[TARGET_COL].plot(ax=axes[0], linewidth=0.7, linestyle="dotted", label="Test Actual")

pd.Series(main_candidate["train_prediction"].values, index=train_df.index, name="Train Fitted").plot(ax=axes[0], linewidth=0.8, label="Train Fitted")
pd.Series(main_candidate["validation_prediction"].values, index=valid_df.index, name="Validation Forecast").plot(ax=axes[0], linewidth=0.8, linestyle="dashed", label="Validation Forecast")
pd.Series(main_candidate["test_prediction"].values, index=test_df.index, name="Test Forecast").plot(ax=axes[0], linewidth=0.8, linestyle="dotted", label="Test Forecast")

axes[0].set_title("Regression Forecast — Actual vs Fitted / Forecast")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Gold Price")
axes[0].legend(loc="best")

# Forecast errors.
pd.Series(train_df[TARGET_COL].values - main_candidate["train_prediction"].values, index=train_df.index, name="Train Residual").plot(ax=axes[1], linewidth=0.7, label="Train Residual")
pd.Series(valid_df[TARGET_COL].values - main_candidate["validation_prediction"].values, index=valid_df.index, name="Validation Residual").plot(ax=axes[1], linewidth=0.7, linestyle="dashed", label="Validation Residual")
pd.Series(test_df[TARGET_COL].values - main_candidate["test_prediction"].values, index=test_df.index, name="Test Residual").plot(ax=axes[1], linewidth=0.7, linestyle="dotted", label="Test Residual")
axes[1].axhline(y=0, linewidth=0.7)
axes[1].set_title("Regression Forecast Errors")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Actual - Predicted")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()

# Coefficient table display.
coef_df = pd.DataFrame(coefficient_table(main_candidate["test_fit"]))
display(coef_df)

# Residual ACF chart, matching professor diagnostics style.
fig, ax = plt.subplots(figsize=(10, 4))
pd.Series(train_valid_residuals, index=train_valid_df.index).plot(ax=ax, linewidth=0.7)
ax.axhline(y=0, linewidth=0.7)
ax.set_title("Main Regression Residuals — Train + Validation Fit")
ax.set_xlabel("Date")
ax.set_ylabel("Residual")
plt.show()

# --------------------------------------------------------------------------------------
# JSON-ready objects produced in CELL 4
# --------------------------------------------------------------------------------------

model_candidate_records = []
for candidate in regression_candidates:
    model_candidate_records.append({
        "model_id": candidate["model_id"],
        "model_name": candidate["model_name"],
        "formula": candidate["formula"],
        "selected_predictors": candidate["selected_predictors"],
        "metrics": candidate["metrics"],
        "fit_statistics_validation_fit": candidate["fit_statistics_validation_fit"],
        "fit_statistics_test_fit": candidate["fit_statistics_test_fit"],
    })

regression_results = {
    "artifact_name": "regression_results",
    "notebook": "06_regression_based_forecasting.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_family": "Regression-Based Forecasting",
    "professor_reference": "Chapter 17 - Regression-based forecasting (2).ipynb",
    "dataset": {
        "dataset_name": "Dataset B — Core Multivariate",
        "target": TARGET_COL,
        "start": CORE_MULTIVARIATE_START,
        "end": CORE_MULTIVARIATE_END,
        "official_forecast_cutoff": OFFICIAL_FORECAST_CUTOFF_DATE,
        "rows_before_missing_drop": int(rows_before_dropna),
        "rows_after_missing_drop": int(rows_after_dropna),
        "rows_dropped_due_to_missing_selected_model_values": int(rows_dropped_missing),
    },
    "splits": {
        "train": {"start": TRAIN_START, "end": TRAIN_END, "rows": int(len(train_df))},
        "validation": {"start": VALIDATION_START, "end": VALIDATION_END, "rows": int(len(valid_df))},
        "test": {"start": TEST_START, "end": TEST_END, "rows": int(len(test_df))},
    },
    "excluded_from_main_regression": EXCLUDED_FROM_MAIN_REGRESSION,
    "selected_predictors_requested": REQUESTED_REGRESSION_PREDICTORS,
    "selected_predictors_used": REGRESSION_PREDICTORS,
    "candidate_models": model_candidate_records,
    "candidate_metric_table": candidate_summary_rows,
    "main_regression_model_id": MAIN_REGRESSION_MODEL_ID,
    "main_regression_model_name": main_candidate["model_name"],
    "main_regression_formula": main_candidate["formula"],
    "main_metrics": main_candidate["metrics"],
    "main_fit_statistics": {
        "validation_fit_train_only": main_candidate["fit_statistics_validation_fit"],
        "test_fit_train_plus_validation": main_candidate["fit_statistics_test_fit"],
    },
    "methodology_notes": [
        "The validation forecast is generated from coefficients fitted on the training period only.",
        "The test forecast is generated from coefficients fitted on training plus validation only.",
        "The test period is not used to estimate regression coefficients.",
        "The main model excludes high_yield because it starts too late for the core multivariate window.",
        "The selected factor set is intentionally limited to keep the regression readable and reduce blind multicollinearity risk.",
    ],
}

regression_diagnostics = {
    "artifact_name": "regression_diagnostics",
    "notebook": "06_regression_based_forecasting.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "diagnostic_model_id": MAIN_REGRESSION_MODEL_ID,
    "diagnostic_fit_scope": "train_plus_validation",
    "fit_statistics": fit_stats(main_candidate["test_fit"]),
    "residual_summary": {
        "train_plus_validation": residual_summary(train_valid_residuals),
        "test": residual_summary(test_residuals),
    },
    "residual_acf": residual_acf_records,
    "ljung_box": ljung_box_records(train_valid_residuals, lags=(10, 20)),
    "vif_table": vif_table,
    "diagnostic_notes": [
        "High VIF values indicate possible multicollinearity and should be discussed, not hidden.",
        "Residual autocorrelation means plain OLS may not fully capture time-series dependence.",
        "Regression is treated as one classical forecasting candidate, not the final selected model.",
    ],
}

regression_coefficients = {
    "artifact_name": "regression_coefficients",
    "notebook": "06_regression_based_forecasting.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MAIN_REGRESSION_MODEL_ID,
    "model_name": main_candidate["model_name"],
    "fit_scope": "train_plus_validation",
    "formula": main_candidate["formula"],
    "coefficient_table": coefficient_table(main_candidate["test_fit"]),
    "significance_rule": "p_value < 0.05 is marked significant_at_0_05",
    "professor_style_fields": [
        "coefficient",
        "std_error",
        "t_value",
        "p_value",
        "r_squared",
        "adjusted_r_squared",
        "f_statistic",
        "f_p_value",
    ],
}

page_regression = {
    "page_title": "Regression-Based Forecasting",
    "page_subtitle": "Professor-style OLS forecasting model using the core multivariate gold dataset.",
    "artifact_source_files": [
        "artifacts/models/regression_results.json",
        "artifacts/models/regression_diagnostics.json",
        "artifacts/interpretability/regression_coefficients.json",
    ],
    "model_status": {
        "model_family": "Regression-Based Forecasting",
        "main_regression_model_id": MAIN_REGRESSION_MODEL_ID,
        "main_regression_model_name": main_candidate["model_name"],
        "official_forecast_cutoff": OFFICIAL_FORECAST_CUTOFF_DATE,
        "high_yield_rule": "high_yield excluded from the main regression because it starts too late for the core multivariate model window.",
    },
    "dataset_window": {
        "dataset_name": "Dataset B — Core Multivariate",
        "start": CORE_MULTIVARIATE_START,
        "end": CORE_MULTIVARIATE_END,
        "target": TARGET_COL,
        "rows_used": int(rows_after_dropna),
    },
    "splits": regression_results["splits"],
    "selected_features": REGRESSION_PREDICTORS,
    "excluded_features": EXCLUDED_FROM_MAIN_REGRESSION,
    "model_explanation": [
        "This page uses ordinary least squares regression as a classical forecasting method.",
        "The model is evaluated with fixed time-based train, validation, and test periods.",
        "Validation performance is estimated using only the training period for fitting.",
        "Test performance is estimated after fitting on training plus validation.",
    ],
    "kpi_cards": {
        "validation": main_candidate["metrics"]["validation_from_train_fit"],
        "test": main_candidate["metrics"]["test_from_train_validation_fit"],
        "fit_statistics": fit_stats(main_candidate["test_fit"]),
    },
    "tables": {
        "candidate_metric_table": candidate_summary_rows,
        "coefficients": regression_coefficients["coefficient_table"],
        "vif": vif_table,
    },
    "charts": {
        "actual_vs_predicted": regression_forecast_path_df[["date", "split", "actual", "predicted"]].to_dict(orient="records"),
        "residuals": regression_forecast_path_df[["date", "split", "residual"]].to_dict(orient="records"),
        "absolute_errors": regression_forecast_path_df[["date", "split", "absolute_error", "absolute_percentage_error"]].to_dict(orient="records"),
    },
    "limitations": [
        "OLS regression can be sensitive to multicollinearity among macroeconomic predictors.",
        "Lagged gold price can dominate level forecasting, so coefficient interpretation should be careful.",
        "Residual autocorrelation may remain because gold prices are time-series data.",
        "This is one forecasting candidate and should be compared against ARIMA, SARIMAX, XGBoost, and other models later.",
    ],
}

print("✅ CELL 4 complete — regression objects ready for export.")


In [ ]:
# ======================================================================================
# CELL 5 — ARTIFACT EXPORT
# ======================================================================================
# Purpose:
# - Export Notebook 06 JSON artifacts for the regression page and future validation notebook.
# ======================================================================================

def json_safe(obj):
    """Convert pandas/numpy/statsmodels objects into JSON-safe Python objects."""
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj) or np.isinf(obj):
            return None
        return float(obj)
    if isinstance(obj, np.ndarray):
        return json_safe(obj.tolist())
    if isinstance(obj, pd.Series):
        return json_safe(obj.to_dict())
    if isinstance(obj, pd.DataFrame):
        return json_safe(obj.to_dict(orient="records"))
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if pd.isna(obj) if not isinstance(obj, (str, bytes, list, dict, tuple)) else False:
        return None
    return obj

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_safe(data), f, indent=2, ensure_ascii=False)
    print("✅ Wrote:", path)

regression_results_path = MODELS_ARTIFACTS_DIR / "regression_results.json"
regression_diagnostics_path = MODELS_ARTIFACTS_DIR / "regression_diagnostics.json"
regression_coefficients_path = INTERPRETABILITY_ARTIFACTS_DIR / "regression_coefficients.json"
page_regression_path = PAGES_ARTIFACTS_DIR / "page_regression.json"

write_json(regression_results_path, regression_results)
write_json(regression_diagnostics_path, regression_diagnostics)
write_json(regression_coefficients_path, regression_coefficients)
write_json(page_regression_path, page_regression)

print("\n✅ Notebook 06 artifacts exported:")
print("-", regression_results_path.relative_to(PROJECT_DIR))
print("-", regression_diagnostics_path.relative_to(PROJECT_DIR))
print("-", regression_coefficients_path.relative_to(PROJECT_DIR))
print("-", page_regression_path.relative_to(PROJECT_DIR))

# Quick verification load.
for path in [regression_results_path, regression_diagnostics_path, regression_coefficients_path, page_regression_path]:
    with open(path, "r", encoding="utf-8") as f:
        loaded = json.load(f)
    print("Verified JSON:", path.name, "| top-level keys:", list(loaded.keys())[:8])


In [ ]:
# ======================================================================================
# CELL 6 — GITHUB PUSH
# ======================================================================================
# Purpose:
# - Commit and push Notebook 06 outputs to GitHub.
# ======================================================================================

files_to_add = [
    "artifacts/models/regression_results.json",
    "artifacts/models/regression_diagnostics.json",
    "artifacts/interpretability/regression_coefficients.json",
    "artifacts/pages/page_regression.json",
]

print("📌 Git status before add:")
run_cmd(["git", "status", "--short"], cwd=str(PROJECT_DIR), allow_fail=True)

run_cmd(["git", "add"] + files_to_add, cwd=str(PROJECT_DIR))

commit_check = run_cmd(
    ["git", "status", "--porcelain"],
    cwd=str(PROJECT_DIR),
    allow_fail=True
).stdout.strip()

if commit_check:
    commit_message = "Add Notebook 06 regression forecasting artifacts"
    run_cmd(["git", "commit", "-m", commit_message], cwd=str(PROJECT_DIR))
    run_cmd(["git", "push", "origin", BRANCH], cwd=str(PROJECT_DIR))
    print("✅ GitHub push complete.")
else:
    print("✅ No changes to commit. Artifacts already match repository.")

print("✅ Notebook 06 complete.")
